# Modern LangChain with Groq — Workshop Notebook

**Focus:** LLM invocations, streaming, messages, structured output, prompt templates, LCEL chains, output parsers, and batch processing.

**Business scenario:** An AI assistant for a technical training company, supporting course advice, learner conversations, corporate workshop enquiries, proposal generation, and learner recommendations.

## Learning outcomes
- Call a Groq-hosted chat model synchronously and asynchronously.
- Stream model output.
- Use role-based messages and conversation context.
- Request structured model output.
- Build reusable prompt templates and LCEL pipelines.
- Use output parsers and batch execution.

## Chapter map

| Chapter | Topic | Real-world practical |
|---|---|---|
| 1 | LLM invocations | Recommend a course to a learner |
| 2 | Streaming | Stream a technical explanation |
| 3 | Messages | Answer a contextual follow-up |
| 4 | Structured output | Extract corporate workshop requirements |
| 5 | Prompt templates | Tailor course descriptions to an audience |
| 6 | Chains and LCEL | Draft a workshop proposal |
| 7 | Output parsers | Produce a validated enquiry summary |
| 8 | Batch processing | Recommend courses to multiple learners |

Each chapter has a scenario, teaching notes, runnable code, and an exercise. Code cells make real API calls when executed; outputs can vary.

## Setup — install dependencies

Run once in the notebook environment. Restart the kernel if requested.

In [ ]:
%pip install -U langchain langchain-core langchain-groq pydantic

## Configure credentials and model

Set `GROQ_API_KEY` in your environment. The cell prompts securely if the variable is missing. Do not commit API keys to source control.

`GROQ_MODEL` can override the default model identifier if it is not available to your Groq account.

In [ ]:
import os
from getpass import getpass

with open(r"C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\key-vault\groq\api.key") as f:
    api_key = f.read().strip()

MODEL_NAME = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL_NAME, api_key=api_key, temperature=0.2)
print(f"Configured ChatGroq with model: {MODEL_NAME}")

# Chapter 1 — LLM Invocations

## Scenario
A learner knows Python basics and SQL and wants to become a data engineer. The assistant recommends the next learning steps.

## APIs
- `ChatGroq(...)`: configures the hosted chat model.
- `invoke(input)`: synchronous call.
- `ainvoke(input)`: asynchronous call.

## Notes
Begin with direct invocation before using LCEL. This makes the model call and its input/output easy to inspect. Use asynchronous calls when an application benefits from non-blocking I/O.

In [ ]:
question = (
    "I know Python basics and SQL. I want to become a data engineer. "
    "Recommend my next three learning steps and explain why."
)
response = llm.invoke(question)
print(response.content)

### Optional extension — asynchronous invocation

Jupyter supports top-level `await`.

In [ ]:
async_response = await llm.ainvoke(
    "Explain the difference between batch and streaming data engineering in simple terms."
)
print(async_response.content)

### Exercise
Change the profile to someone who knows JavaScript and wants to learn Generative AI. Ask for a three-step plan.

**Observe:** Responses are generated by the hosted model and may differ between runs.

# Chapter 2 — Model Output and Streaming

## Scenario
A learner asks for an explanation of RAG. The application should display the response as it arrives rather than waiting for the full answer.

## APIs
- `stream(input)`: yields response chunks synchronously.
- `astream(input)`: yields chunks asynchronously.

## Notes
Streaming improves perceived responsiveness, but does not guarantee a shorter total generation time. Chunk boundaries may not align with words or sentences.

In [ ]:
stream_prompt = (
    "Explain Retrieval-Augmented Generation (RAG) to a software developer. "
    "Use a short analogy and three practical steps."
)
for chunk in llm.stream(stream_prompt):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()

### Optional extension — asynchronous streaming

In [ ]:
async for chunk in llm.astream(
    "Explain embeddings and cosine similarity in simple terms."
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()

### Exercise
Stream an explanation of vector databases. Compare beginner-friendly and advanced prompts.

# Chapter 3 — Messages and Conversations

## Scenario
A learner was advised to start with SQL and data modelling. They have completed those topics and ask what to learn next.

## APIs
- `SystemMessage`: sets assistant behaviour.
- `HumanMessage`: represents user input.
- `AIMessage`: represents an earlier assistant response.
- `invoke(messages)`: sends role-labelled messages to the model.

## Notes
This example passes history explicitly. It does not persist conversation history; an application must store and supply it when needed.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a practical technical-course advisor. Give concise, actionable guidance."),
    HumanMessage(content="I know Python and want to become a data engineer. What should I learn first?"),
    AIMessage(content="Start with SQL, relational modelling, and data quality fundamentals."),
    HumanMessage(content="I have completed those. What should I learn next?"),
]
message_response = llm.invoke(messages)
print(message_response.content)

### Exercise
Add another prior assistant message and a follow-up asking how to practise with a project.

**Discussion:** What context is lost if only the latest user message is sent?

# Chapter 4 — Structured Output

## Scenario
A corporate client sends a free-form workshop enquiry. The training team needs consistent fields for follow-up.

## APIs
- Pydantic `BaseModel`: defines the expected data shape.
- `with_structured_output(Schema)`: requests data matching the schema.
- `invoke(text)`: calls the model and returns a structured result when supported.

## Notes
Structured output does not replace business validation. Check required fields, allowed values, and uncertain or missing information before using the result operationally.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class WorkshopEnquiry(BaseModel):
    technology: str = Field(description="Main technology requested")
    participant_count: int | None = Field(description="Number of participants, if stated")
    skill_level: Literal["beginner", "intermediate", "advanced", "unspecified"]
    duration_days: int | None = Field(description="Duration in days, if stated")
    delivery_mode: Literal["online", "in_person", "hybrid", "unspecified"]
    business_goal: str = Field(description="What the client wants participants to achieve")

structured_llm = llm.with_structured_output(WorkshopEnquiry)

enquiry = """
We need a two-day intermediate LangChain workshop for 18 developers.
Prefer in-person delivery in Bengaluru. The team should learn to build API-based
LLM applications and simple RAG pipelines.
"""
parsed_enquiry = structured_llm.invoke(enquiry)
print(parsed_enquiry)
print("Technology:", parsed_enquiry.technology)
print("Participants:", parsed_enquiry.participant_count)

### Exercise
Change the enquiry to a one-day beginner Python workshop for 25 college students.

**Note:** Structured-output support depends on the chosen model and integration. If unsupported, choose a compatible Groq model.

# Chapter 5 — Prompt Templates

## Scenario
The marketing team needs tailored course descriptions for students, working professionals, and corporate clients.

## APIs
- `ChatPromptTemplate.from_template(...)`: creates a reusable prompt.
- `invoke({...})`: fills template variables.
- `StrOutputParser`: extracts text from the model message.

## Notes
This is the first chapter where LCEL is the main pattern: `prompt | llm | parser`. Each step passes its output to the next step.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

course_prompt = ChatPromptTemplate.from_template(
    "You are a technical training content writer. "
    "Write a course description for: {course}. "
    "Target audience: {audience}. "
    "Length: about 100 words. Include learning outcomes and prerequisites."
)
course_description_chain = course_prompt | llm | StrOutputParser()

description = course_description_chain.invoke({
    "course": "Generative AI application development with Python",
    "audience": "software developers with basic Python knowledge",
})
print(description)

### Exercise
Generate another description for corporate engineering managers. Keep the course constant and change only the audience.

**Discussion:** Why is a template easier to maintain than duplicating a large prompt string?

# Chapter 6 — Chains and LCEL Composition

## Scenario
A client requests a workshop proposal. The coordinator needs learning objectives and a concise proposal summary.

## APIs
- LCEL composition using `|`.
- `RunnableParallel`: runs multiple branches from the same input.
- `RunnableLambda`: wraps a Python function as a Runnable.
- `RunnablePassthrough`: passes input through unchanged.
- `StrOutputParser`: extracts text from model messages.

## Notes
Use LCEL for readable composition. Parallel branches are useful when outputs are independent, but they may increase API usage and cost.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

proposal_input = {
    "client_need": (
        "A two-day workshop for intermediate Python developers who need to build "
        "LLM-powered internal support assistants. Include tool calling and basic RAG."
    )
}

objectives_prompt = ChatPromptTemplate.from_template(
    "For this client requirement, produce five measurable workshop learning objectives. "
    "Requirement: {client_need}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Write a concise proposal summary (about 120 words). Mention audience, scope, and practical outcome. "
    "Requirement: {client_need}"
)

objectives_chain = objectives_prompt | llm | StrOutputParser()
summary_chain = summary_prompt | llm | StrOutputParser()

proposal_chain = RunnableParallel(
    original_request=RunnablePassthrough(),
    learning_objectives=objectives_chain,
    proposal_summary=summary_chain,
)
proposal_result = proposal_chain.invoke(proposal_input)

print("LEARNING OBJECTIVES\n")
print(proposal_result["learning_objectives"])
print("\nPROPOSAL SUMMARY\n")
print(proposal_result["proposal_summary"])

### Optional extension — add Python post-processing

In [ ]:
def add_word_count(text: str) -> dict:
    return {"text": text, "word_count": len(text.split())}

summary_with_count = summary_chain | RunnableLambda(add_word_count)
summary_stats = summary_with_count.invoke(proposal_input)
print("Word count:", summary_stats["word_count"])
print(summary_stats["text"])

### Exercise
Add a third parallel branch that proposes three hands-on lab titles.

**Discussion:** When is sequential composition more appropriate than parallel branches?

# Chapter 7 — Output Parsers

## Scenario
The operations team wants a training enquiry summarised in a predictable schema for downstream Python processing.

## APIs
- `PydanticOutputParser`: supplies formatting instructions and parses output into a Pydantic model.
- `StrOutputParser`: extracts plain text from a model message.
- LCEL: composes prompt, model, and parser.

## Notes
This chapter contrasts parser-based formatting with Chapter 4's `with_structured_output()`. The parser approach makes formatting instructions and parsing explicit. Handle parsing and validation errors in production.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class EnquirySummary(BaseModel):
    client_type: str = Field(description="For example, college, startup, or enterprise")
    requested_topic: str = Field(description="Main training topic")
    urgency: str = Field(description="Stated urgency; use 'not specified' if absent")
    next_action: str = Field(description="One practical follow-up action")

summary_parser = PydanticOutputParser(pydantic_object=EnquirySummary)
parser_prompt = ChatPromptTemplate.from_template(
    "Summarise this training enquiry using the requested schema. Do not invent facts.\n"
    "{format_instructions}\n\nEnquiry:\n{enquiry}"
)
parser_chain = parser_prompt | llm | summary_parser

enquiry_summary = parser_chain.invoke({
    "format_instructions": summary_parser.get_format_instructions(),
    "enquiry": (
        "We are a startup with 12 engineers interested in an API-based Groq and "
        "LangChain workshop. We hope to schedule it next month."
    ),
})
print(enquiry_summary)
print("Next action:", enquiry_summary.next_action)

### Exercise
Add a `participant_count` field to the Pydantic model, update its description, and rerun.

**Production note:** Catch validation errors and decide whether to retry, ask for clarification, or route the enquiry to manual review.

# Chapter 8 — Batch Processing

## Scenario
The training team has multiple prospective learners and wants a course recommendation for each.

## APIs
- `batch(inputs)`: executes a Runnable for multiple inputs.
- `abatch(inputs)`: asynchronous batch execution.
- LCEL: reuses a prompt-model-parser chain for each input.

## Notes
Batching is convenient for independent inputs. Consider provider rate limits, concurrency, token usage, and partial failures. Each learner profile is processed as an input to the chain.

In [ ]:
recommendation_prompt = ChatPromptTemplate.from_template(
    "You are a technical learning advisor. Recommend the next two learning steps for this learner. "
    "Keep the response under 80 words. Learner profile: {profile}"
)
recommendation_chain = recommendation_prompt | llm | StrOutputParser()

learner_profiles = [
    {"profile": "Knows Python basics and SQL; wants to become a data engineer."},
    {"profile": "Java developer; wants to build applications using LLM APIs."},
    {"profile": "Data analyst comfortable with pandas; wants to learn machine learning."},
    {"profile": "Frontend developer; wants to learn backend APIs and deployment."},
]

batch_results = recommendation_chain.batch(learner_profiles)
for index, (learner, recommendation) in enumerate(zip(learner_profiles, batch_results), start=1):
    print(f"LEARNER {index}: {learner['profile']}")
    print(recommendation)
    print("-" * 70)

### Optional extension — asynchronous batch

Run this cell in Jupyter; it uses the same chain and inputs.

In [ ]:
async_batch_results = await recommendation_chain.abatch(learner_profiles)
for index, result in enumerate(async_batch_results, start=1):
    print(f"Async result {index}:\n{result}\n")

### Exercise
Add two learner profiles. Compare sequential execution with `batch()` and discuss how rate limits or partial failures should be handled.

# Workshop Recap

| Concept | Main takeaway |
|---|---|
| Direct invocation | Use `invoke()` / `ainvoke()` for straightforward calls |
| Streaming | Use `stream()` / `astream()` to display partial output |
| Messages | Supply role-labelled messages to provide context |
| Structured output | Request data matching a schema, then validate it |
| Prompt templates | Keep prompts reusable and parameterised |
| LCEL | Compose Runnables with `|`; use parallel branches when appropriate |
| Output parsers | Convert model messages into text or validated objects |
| Batch processing | Reuse a chain for multiple independent inputs |

## Troubleshooting
- Confirm `GROQ_API_KEY` is set correctly.
- Confirm `GROQ_MODEL` names a model currently available to your Groq account.
- Check installed versions if an import or method is unavailable.
- Model output is non-deterministic; responses are illustrative, not fixed assertions.
- Keep credentials out of source control.
- For production, add logging, retries, error handling, rate-limit handling, and evaluation.

## Suggested next workshop
Continue with document loading, text splitting, embeddings, vector stores, retrievers, and RAG — then move to tools, agents, and LangGraph workflows.